<a href="https://colab.research.google.com/github/sajina464/C-Programs/blob/main/day_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully!")


Libraries imported successfully!


In [2]:

financial_documents = [
    {
        "id": "D1",
        "text": (
            "Pay credit card bills and loan EMIs on time. "
            "Late payments can negatively affect credit scores."
        )
    },
    {
        "id": "D2",
        "text": (
            "Keeping credit utilization low is generally helpful. "
            "Around 30 percent or less is a commonly used guideline. "
            "Lower utilization does not guarantee a higher score."
        )
    },
    {
        "id": "D3",
        "text": (
            "Loan approval depends on lender requirements, income, "
            "credit history, existing debts and other eligibility criteria. "
            "A credit score alone cannot guarantee loan approval."
        )
    },
    {
        "id": "D4",
        "text": (
            "Review credit reports for incorrect accounts or payment records. "
            "Dispute errors with the relevant credit bureau and provide "
            "supporting documents."
        )
    },
    {
        "id": "D5",
        "text": (
            "Create a budget, track spending and prioritize essential "
            "payments. An emergency fund can help cover unexpected expenses."
        )
    }
]

print("Reference documents:", len(financial_documents))


Reference documents: 5


In [3]:

test_responses = [
    {
        "query": "How can I improve my credit score?",
        "response": (
            "Pay credit card bills on time. "
            "Keep credit utilization low. "
            "Take a new personal loan immediately to increase your score."
        )
    },
    {
        "query": "How can I improve loan eligibility?",
        "response": (
            "Review your existing debts and income. "
            "Loan approval depends on lender requirements. "
            "Every applicant with a score above 750 is guaranteed approval."
        )
    },
    {
        "query": "What should I do about credit report errors?",
        "response": (
            "Check your credit report for errors. "
            "Dispute incorrect records with the relevant credit bureau "
            "and provide supporting documents."
        )
    },
    {
        "query": "How should I manage my monthly money?",
        "response": (
            "Create a budget and track spending. "
            "Invest all your savings in one stock for guaranteed profit."
        )
    }
]

print("Responses ready for evaluation:", len(test_responses))


Responses ready for evaluation: 4


In [4]:

evidence_sentences = []

for doc in financial_documents:
    sentences = re.split(r"(?<=[.!?])\s+", doc["text"])
    for sentence in sentences:
        if sentence.strip():
            evidence_sentences.append({
                "document_id": doc["id"],
                "text": sentence.strip()
            })

evidence_texts = [e["text"] for e in evidence_sentences]

# Fit one vectorizer for both evidence and test recommendations
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2)
)

evidence_vectors = vectorizer.fit_transform(evidence_texts)

recommendation_pattern = re.compile(
    r"\b(should|must|recommend|recommendation|"
    r"consider|avoid|pay|keep|reduce|increase|"
    r"take|invest|create|track|review|dispute|"
    r"prioritize|guaranteed|guarantee)\b",
    re.IGNORECASE
)

def check_recommendations(response, threshold=0.20):
    sentences = re.split(r"(?<=[.!?])\s+", response.strip())
    flagged = []

    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue

        # Check recommendations and strong financial claims
        if not recommendation_pattern.search(sentence):
            continue

        sentence_vector = vectorizer.transform([sentence])
        scores = cosine_similarity(
            sentence_vector, evidence_vectors
        )[0]

        best_index = scores.argmax()
        best_score = float(scores[best_index])

        if best_score < threshold:
            status = "UNSUPPORTED - REVIEW"
        else:
            status = "POTENTIALLY SUPPORTED - VERIFY"

        flagged.append({
            "recommendation": sentence,
            "status": status,
            "similarity_score": round(best_score, 3),
            "closest_evidence_id": (
                evidence_sentences[best_index]["document_id"]
            ),
            "closest_evidence": evidence_sentences[best_index]["text"]
        })

    return flagged


all_results = []

for item in test_responses:
    findings = check_recommendations(item["response"])

    for finding in findings:
        all_results.append({
            "query": item["query"],
            **finding
        })

results_df = pd.DataFrame(all_results)

if not results_df.empty:
    display(results_df)
else:
    print("No recommendation-like sentences detected.")


,query,recommendation,status,similarity_score,closest_evidence_id,closest_evidence
0,How can I improve my credit score?,Pay credit card bills on time.,POTENTIALLY SUPPORTED - VERIFY,0.783,D1,Pay credit card bills and loan EMIs on time.
1,How can I improve my credit score?,Keep credit utilization low.,POTENTIALLY SUPPORTED - VERIFY,0.632,D2,Keeping credit utilization low is generally he...
2,How can I improve my credit score?,Take a new personal loan immediately to increa...,POTENTIALLY SUPPORTED - VERIFY,0.438,D3,A credit score alone cannot guarantee loan app...
3,How can I improve loan eligibility?,Review your existing debts and income.,POTENTIALLY SUPPORTED - VERIFY,0.388,D3,"Loan approval depends on lender requirements, ..."
4,How can I improve loan eligibility?,Every applicant with a score above 750 is guar...,POTENTIALLY SUPPORTED - VERIFY,0.466,D3,A credit score alone cannot guarantee loan app...
5,What should I do about credit report errors?,Dispute incorrect records with the relevant cr...,POTENTIALLY SUPPORTED - VERIFY,0.819,D4,Dispute errors with the relevant credit bureau...
6,How should I manage my monthly money?,Create a budget and track spending.,POTENTIALLY SUPPORTED - VERIFY,0.742,D5,"Create a budget, track spending and prioritize..."
7,How should I manage my monthly money?,Invest all your savings in one stock for guara...,UNSUPPORTED - REVIEW,0.000,D1,Pay credit card bills and loan EMIs on time.


In [5]:

if not results_df.empty:
    unsupported_count = (
        results_df["status"] == "UNSUPPORTED - REVIEW"
    ).sum()

    supported_count = (
        results_df["status"] == "POTENTIALLY SUPPORTED - VERIFY"
    ).sum()

    print("Total recommendations checked:", len(results_df))
    print("Flagged for unsupported evidence:", unsupported_count)
    print("Potentially supported; verification needed:", supported_count)

    results_df.to_csv(
        "unsupported_financial_recommendations.csv",
        index=False
    )

    print("Evaluation report saved successfully!")

else:
    print("No recommendations available to export.")


Total recommendations checked: 8
Flagged for unsupported evidence: 1
Potentially supported; verification needed: 7
Evaluation report saved successfully!


In [6]:

from google.colab import files

files.download("unsupported_financial_recommendations.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>